# Exercise — Choose a Governance Model for Trailhead Provisions

**Trailhead Provisions** grew by acquisition: four domain teams (E-commerce Orders, Customer
Identity & Loyalty, Inventory & Fulfillment, Marketing & Campaigns) own their pipelines and
definitions on a shared platform. A **GDPR audit** and a new **CSRD sustainability mandate**
now create obligations that cross every domain. Decide how to govern. See `INSTRUCTIONS.md`.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog

gc = GovernedCatalog("trailhead.db")
pd.DataFrame([{"domain": gc.catalog.table(t).domain, "table": t,
               "owner": gc.catalog.table(t).owner or "(no owner set)"}
              for t in gc.tables()]).sort_values("domain").reset_index(drop=True)

## 1a. Governance-model decision
Choose **centralized**, **decentralized**, or **federated**, and state which decisions stay central even within your choice.

### Decision — federated governance with a thin central core

Trailhead's domains own their data products and will resist a central team rewriting them, so
**centralized** governance stalls on friction. **Decentralized** is what they have today — and it
produced the audit failure (no shared consent record, no metadata standard, no lineage).
**Federated** keeps domain ownership while making a small set of decisions non-negotiable and
central:

- **PII classification & tagging standard** — central; GDPR exposure is platform-wide.
- **Consent system of record** — central; loyalty and marketing disagree today (GDPR risk).
- **Contract schema & required governance fields** — central schema, domain-authored content.
- **Shared metric definitions** (per-order carbon) — central; CSRD needs one auditable number.

Everything else — schema evolution within a product, pipeline implementation, quality-rule
authoring — stays with the domains. This is the least centralization that satisfies both
regulators without seizing day-to-day control.

## 1b. RACI (governance functions × domains)
One **accountable (A)** owner per function; name the enforcement authority and escalation path.

| Governance function | E-comm | Customer | Inventory | Marketing | Accountable (single) |
|---|---|---|---|---|---|
| PII classification & tagging | R | R | R | R | **Central Governance Lead** |
| Consent system of record | C | R | I | C | **Customer Identity Lead** |
| Data contracts (author) | R | R | R | R | **Owning domain lead (per product)** |
| Data quality SLOs | R | R | R | R | **Owning domain lead** |
| Lineage instrumentation | C | C | R | C | **Platform Engineering** |
| Carbon metric definition | I | I | R | C | **CSRD Reporting Owner** |
| Access policy (Lake Formation) | C | R | C | C | **Central Governance Lead** |

**Enforcement:** the Central Governance Lead can block a product from publishing if it fails the
classification or contract gate. **Escalation:** unresolved cross-domain conflicts (e.g.,
consent) go to a Data Governance Council chaired by the Lead, with the affected domain leads as
voting members.

## 1c. Federated topology
Draw the central core, the four domains, and the key cross-domain flows as a Mermaid diagram.

```mermaid
graph TD
    GC[Central Governance Core<br/>classification • consent SoR • contract schema • metrics]
    subgraph Domains
        D1[E-commerce Orders]
        D2[Customer Identity & Loyalty]
        D3[Inventory & Fulfillment]
        D4[Marketing & Campaigns]
    end
    GC -- standards & gates --> D1 & D2 & D3 & D4
    D2 -- consent SoR --> D4
    D1 -- order events --> D3
    D3 -- carbon source --> R[CSRD Quarterly Report]
    D1 -- order carbon --> R
```